# Factor Mining Workflow with NeMo Agent Toolkit

In this notebook, we showcase how the NeMo Agent toolkit can be used to create an end-to-end factor mining workflow for quantitative finance. This workflow demonstrates how to leverage LLMs to automatically generate, code, and evaluate alpha factors.

## What You'll Learn

By the end of this notebook, you will understand how to:
- Set up a multi-step sequential workflow using NAT
- Implement custom functions for factor generation, code generation, and evaluation
- Use LLMs for creative factor design and code synthesis
- Evaluate factor performance using rank IC metrics
- Run complete end-to-end factor mining workflows with optimization loops

## Workflow Overview

This factor mining workflow consists of four main components:

1. **Factor Generator**: Uses an LLM to generate creative factor descriptions based on market data
2. **Code Generator**: Converts factor descriptions into executable Python code
3. **Rank IC Evaluator**: Evaluates the predictive power of generated factors using Rank IC (Information Coefficient)
4. **Factor Optimization Agent**: Orchestrates the optimization loop, providing feedback and iteratively improving factors

> **Note:** _This example demonstrates the capabilities of NeMo Agent Toolkit for automating quantitative research workflows. The generated factors should be thoroughly validated before use in production._

## Table of Contents

- [0) Setup](#setup)
  - [0.1) Prerequisites](#prereqs)
  - [0.2) API Keys](#api-keys)
  - [0.3) Installing Dependencies](#installing-deps)
- [1) Understanding Factor Mining](#understanding-workflow)
  - [1.1) What is Factor Mining?](#what-is-factor-mining)
  - [1.2) Workflow Architecture](#workflow-architecture)
- [2) Source Code Components](#source-code)
  - [2.1) Factor Generator](#factor-generator)
  - [2.2) Factor Code Generator](#factor-code-generator)
  - [2.3) Factor Evaluator](#factor-evaluator) *(includes Rank IC utilities)*
  - [2.4) Factor Mining Optimization Workflow](#optimization-agent)
  - [2.5) Register Module](#register-module)
- [3) Configuration](#configuration)
  - [3.1) Workflow Configuration](#workflow-config)
  - [3.2) Operator Templates](#operator-templates)
- [4) Running the Workflow](#running-workflow)
  - [4.1) Via Command Line](#run-cli)
  - [4.2) Programmatically in Python](#run-python)
- [5) Interpreting Results](#interpreting-results)
  - [5.1) Understanding Evaluation Metrics](#evaluation-metrics)
  - [5.2) Analyzing Generated Factors](#analyzing-factors)
- [6) Next Steps](#next-steps)

<a id="setup"></a>
# 0) Setup

<a id="prereqs"></a>
## 0.1) Prerequisites

- **Platform:** Linux, macOS, or Windows
- **Python:** version 3.11, 3.12, or 3.13
- **Package manager:** pip or uv

<a id="api-keys"></a>
## 0.2) API Keys

For this notebook, you will need an NVIDIA API key. Get yours from [build.nvidia.com](https://build.nvidia.com/settings/api-keys)

In [1]:
import getpass
import os

if "NVIDIA_API_KEY" not in os.environ:
    nvidia_api_key = getpass.getpass("Enter your NVIDIA API key: ")
    os.environ["NVIDIA_API_KEY"] = nvidia_api_key

<a id="installing-deps"></a>
## 0.3) Installing Dependencies

Install the factor mining workflow package:

In [ ]:
# Install the factor mining workflow package
# Run this from the notebooks directory (one level up to project root)
%pip install -e ..

Obtaining file:///home/scratch.phuo_wwfo_1/github/factor-mining-agent
  DEPRECATION: Setting PIP_CONSTRAINT will not affect build constraints in the future, pip 26.2 will enforce this behaviour change. A possible replacement is to specify build constraints using --build-constraint or PIP_BUILD_CONSTRAINT. To disable this warning without any build constraints set --use-feature=build-constraint or PIP_USE_FEATURE="build-constraint".
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for factor_mining_workflow (pyproject.toml) ... done
  Created wheel for factor_mining_workflow: filename=factor_mining_workflow-0.1.dev2+g1f00969e8.d20260210-0.editable-py3-none-any.whl size=2142 sha256=185a0d718a0de47edc8c2d2959ff33e60c39990474357c6e9e432790235fd5bb
  Stored in directory: /tmp/pip-ephem-wheel-cache-4wszuhkt/wheels/e9

<a id="understanding-workflow"></a>
# 1) Understanding Factor Mining

<a id="what-is-factor-mining"></a>
## 1.1) What is Factor Mining?

Factor mining is the process of discovering quantitative signals (factors) that have predictive power for future stock returns. In quantitative finance, factors are numerical characteristics of stocks that can be used to:

- **Predict returns**: Factors with high Information Coefficient (IC) can help forecast which stocks will outperform
- **Construct portfolios**: Long stocks with favorable factor values, short those with unfavorable values
- **Manage risk**: Understand exposures to different sources of returns

Traditional factor research is labor-intensive, requiring researchers to:
1. Formulate hypotheses about what might predict returns
2. Implement the factor calculation in code
3. Backtest the factor's performance
4. Iterate based on results

This workflow automates this process using LLMs!

<a id="workflow-architecture"></a>
## 1.2) Workflow Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                  Factor Optimization Agent                      │
│                                                                 │
│  ┌─────────────┐    ┌─────────────┐    ┌─────────────────────┐  │
│  │   Factor    │    │    Code     │    │    Rank IC          │  │
│  │  Generator  │───▶│  Generator  │───▶│    Evaluator        │  │
│  │   (LLM)     │    │   (LLM)     │    │  (Statistical)      │  │
│  └─────────────┘    └─────────────┘    └──────────┬──────────┘  │
│                                                   │             │
│                         ┌─────────────────────────┘             │
│                         │                                       │
│                         ▼                                       │
│              ┌─────────────────────┐                            │
│              │  IC Good? Accept!   │                            │
│              │  IC Poor? Optimize  │◀────────────┐              │
│              └──────────┬──────────┘             │              │
│                         │                        │              │
│                         ▼                        │              │
│              ┌─────────────────────┐             │              │
│              │   Optimization      │─────────────┘              │
│              │   Feedback (LLM)    │                            │
│              └─────────────────────┘                            │
└─────────────────────────────────────────────────────────────────┘
```

The workflow uses a **closed-loop optimization** approach:
1. Generate factor ideas using an LLM
2. Convert ideas to executable Python code
3. Evaluate the factor's predictive power (Rank IC)
4. If IC meets threshold → Accept and save
5. If IC is poor → Generate optimization advice and retry

<a id="source-code"></a>
# 2) Source Code Components

Below we load and display the source code for each component of the factor mining workflow. Use `%load` to load the Python source files.

<a id="factor-generator"></a>
## 2.1) Factor Generator

The Factor Generator uses an LLM to create quantitative factor descriptions based on available price-volume data and predefined operators.

In [ ]:
# %load ../src/factor_mining_workflow/factor_generator.py


<a id="factor-code-generator"></a>
## 2.2) Factor Code Generator

The Factor Code Generator converts factor descriptions (JSON) into executable Python code. It uses an LLM to synthesize the factor function and automatically includes the required operator implementations from `calculator.json`.


In [ ]:
# %load ../src/factor_mining_workflow/factor_code_generator.py


<a id="factor-evaluator"></a>
## 2.3) Factor Evaluator

The Factor Evaluator computes the Spearman correlation (Rank IC) between factor values and forward stock returns to measure predictive power. It consolidates all IC calculation and evaluation utilities.

In [ ]:
# %load ../src/factor_mining_workflow/factor_evaluator.py

<a id="optimization-agent"></a>
## 2.4) Factor Mining Optimization Workflow

The Factor Mining Optimization Workflow orchestrates the entire optimization loop, generating factors, evaluating them, and providing feedback for improvement. It composes the factor generator, code generator, and evaluator modules.

In [ ]:
# %load ../src/factor_mining_workflow/factor_mining_optimization_workflow.py

<a id="register-module"></a>
## 2.5) Register Module

The register module imports all components to make them available to the NeMo Agent Toolkit.

In [ ]:
# %load ../src/factor_mining_workflow/register.py

<a id="configuration"></a>
# 3) Configuration

<a id="workflow-config"></a>
## 3.1) Workflow Configuration

The workflow configuration defines how the factor optimization agent operates:

In [ ]:
# Display the configuration file
with open("../configs/config-optimization.yml", "r") as f:
    print(f.read())

# Factor Mining Workflow
# Iteratively generates and optimizes quantitative factors:
# 1. Generate factor descriptions using LLM
# 2. Convert to executable Python code
# 3. Evaluate rank IC against forward returns
# 4. Accept if IC meets threshold, otherwise retry with feedback

llms:
  nim:
    _type: nim
    model_name: meta/llama-3.1-70b-instruct
    temperature: 0.7
    max_tokens: 4096

workflow:
  _type: factor_optimizer
  llm_name: nim
  ic_threshold: 0.03        # Minimum |IC| to accept
  p_value_threshold: 0.2    # Maximum p-value for significance
  max_iterations: 1         # Optimization attempts
  num_factors: 5            # Factors per iteration
  forward_periods: 5        # Days for forward returns
  save_results: true        # Save results to disk
  use_colors: true          # ANSI colors in output



### Configuration Parameters Explained

| Parameter | Description |
|-----------|-------------|
| `llm_name` | The LLM to use for factor generation and optimization advice |
| `ic_threshold` | Minimum absolute IC value to accept a factor (e.g., 0.03 = 3%) |
| `p_value_threshold` | Maximum p-value for statistical significance (e.g., 0.1 = 10%) |
| `max_iterations` | Maximum number of optimization iterations before accepting best result |
| `num_factors` | Number of factors to generate per iteration |
| `forward_periods` | Number of days for forward return calculation (e.g., 5 = weekly) |
| `save_results` | Whether to save successful factors to disk |

<a id="operator-templates"></a>
## 3.2) Operator Templates

The workflow uses predefined operators from `calculator.json`. Here's a sample of available operators:

In [ ]:
import json
from pathlib import Path

# Load and display sample operators
template_path = Path("../src/factor_mining_workflow/template/calculator.json")
with open(template_path, "r") as f:
    operators = json.load(f)

print(f"Total operators available: {len(operators)}\n")
print("Sample operators:")
print("-" * 60)
for op in operators[:10]:
    print(f"\n{op['name']}:")
    print(f"  Signature: {op['signature']}")
    print(f"  Meaning: {op['meanings'][:80]}...")

Total operators available: 66

Sample operators:
------------------------------------------------------------

Add:
  Signature: Add(x: pd.DataFrame, y: pd.DataFrame) -> pd.DataFrame
  Meaning: Add two DataFrames element-wise and return the result....

Sub:
  Signature: Sub(x: pd.DataFrame, y: pd.DataFrame) -> pd.DataFrame
  Meaning: Subtract the second DataFrame from the first DataFrame element-wise and return t...

Mul:
  Signature: Mul(x: pd.DataFrame, y: pd.DataFrame) -> pd.DataFrame
  Meaning: Multiply two DataFrames element-wise and return the result....

Div:
  Signature: Div(x: pd.DataFrame, y: pd.DataFrame) -> pd.DataFrame
  Meaning: Divide the first DataFrame by the second DataFrame element-wise and return the r...

Rank_Add:
  Signature: Rank_Add(x: pd.DataFrame, y: pd.DataFrame) -> pd.DataFrame
  Meaning: Return the sum of the quantiles of the elements in DataFrame x and the correspon...

Rank_Sub:
  Signature: Rank_Sub(x: pd.DataFrame, y: pd.DataFrame) -> pd.DataFrame
  Me

<a id="running-workflow"></a>
# 4) Running the Workflow
Now we may run the workflow and see the generated factors. 

In [ ]:
# Run the factor optimization workflow via CLI
# Note: Run from the project root directory (one level up)
%cd ..
!nat run --config_file configs/config-optimization.yml --input "momentum factors"

2026-02-10 02:05:10 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'configs/config-optimization.yml'
2026-02-10 02:05:12 - INFO     - factor_mining_workflow.rank_ic_evaluator:47 - Loaded Open.csv with shape (3270, 392)
2026-02-10 02:05:12 - INFO     - factor_mining_workflow.rank_ic_evaluator:47 - Loaded Close.csv with shape (3270, 392)
2026-02-10 02:05:12 - INFO     - factor_mining_workflow.rank_ic_evaluator:47 - Loaded High.csv with shape (3270, 392)
2026-02-10 02:05:13 - INFO     - factor_mining_workflow.rank_ic_evaluator:47 - Loaded Low.csv with shape (3270, 392)
2026-02-10 02:05:13 - INFO     - factor_mining_workflow.rank_ic_evaluator:47 - Loaded Volume.csv with shape (3270, 392)

Configuration Summary:
--------------------
Workflow Type: factor_optimizer
Number of Functions: 0
Number of Function Groups: 0
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Au

### Running with Different Factor Types

In [ ]:
# Run the factor optimization workflow via CLI (already in project root from previous cell)
!nat run --config_file configs/config-optimization.yml --input "volatility factors"

2026-02-10 02:17:26 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'configs/config-optimization.yml'
2026-02-10 02:17:26 - INFO     - nat.tool.code_execution.register:53 - [DEBUG] Created sandbox of type: local
2026-02-10 02:17:26 - INFO     - factor_mining_workflow.rank_ic_evaluator:47 - Loaded Open.csv with shape (3270, 392)
2026-02-10 02:17:26 - INFO     - factor_mining_workflow.rank_ic_evaluator:47 - Loaded Close.csv with shape (3270, 392)
2026-02-10 02:17:26 - INFO     - factor_mining_workflow.rank_ic_evaluator:47 - Loaded High.csv with shape (3270, 392)
2026-02-10 02:17:27 - INFO     - factor_mining_workflow.rank_ic_evaluator:47 - Loaded Low.csv with shape (3270, 392)
2026-02-10 02:17:27 - INFO     - factor_mining_workflow.rank_ic_evaluator:47 - Loaded Volume.csv with shape (3270, 392)

Configuration Summary:
--------------------
Workflow Type: factor_optimizer
Number of Functions: 1
Number of Function Groups: 0
Number of LLMs: 1
Number of Embedders: 0
N

In [ ]:
# Run the factor optimization workflow via CLI (already in project root from previous cell)
!nat run --config_file configs/config-optimization.yml --input "mean-reversion factors"

<a id="interpreting-results"></a>
# 5) Interpreting Results

<a id="evaluation-metrics"></a>
## 5.1) Understanding Evaluation Metrics

The workflow evaluates factors using several key metrics:

| Metric | Description | Good Value |
|--------|-------------|------------|
| **Mean IC** | Average Spearman correlation between factor and forward returns | \|IC\| > 0.03 |
| **IC Std** | Standard deviation of IC values | Lower is more consistent |
| **IC IR** | Information Ratio = Mean IC / IC Std | > 0.5 is good |
| **T-statistic** | Statistical significance of mean IC | \|t\| > 2 is significant |
| **P-value** | Probability IC is different from zero | < 0.05 is significant |
| **Positive IC Ratio** | Fraction of periods with positive IC | > 0.55 is good |

### Rank IC Value Interpretation

| Rank IC Value | Interpretation |
|---------------|----------------|
| **< 0.02** | **Weak/Noise**: The factor has very little predictive power. It may still be useful when combined in a multi-factor model, but it's not a strong standalone signal. |
| **0.02 – 0.05** | **Good/Standard**: This is the target range for many institutional-grade factors (like Value or Quality). While it seems low, when applied across a large universe of stocks, it generates consistent Alpha. |
| **0.05 – 0.15** | **Strong**: This indicates a very high-quality signal, typical of successful short-term Momentum or specialized High-Frequency factors. |
| **> 0.15** | **Exceptional/Suspicious**: In a live market, a Rank IC this high is rare. If your agent reports this, you should check for look-ahead bias (accidentally using future data to predict the past) or overfitting. |

In [8]:
import json

def interpret_ic_results(result_json: str):
    """
    Interpret the IC evaluation results.
    """
    result = json.loads(result_json)
    
    print("=" * 60)
    print("FACTOR EVALUATION RESULTS")
    print("=" * 60)
    
    status = result.get("status", "unknown")
    print(f"\nStatus: {status.upper()}")
    print(f"Iterations: {result.get('iterations', 'N/A')}")
    
    metrics = result.get("evaluation_metrics", {})
    if metrics:
        print("\nEvaluation Metrics:")
        print("-" * 40)
        print(f"  Mean IC: {metrics.get('mean_ic', 'N/A'):.4f}" if metrics.get('mean_ic') else "  Mean IC: N/A")
        print(f"  IC Std: {metrics.get('ic_std', 'N/A'):.4f}" if metrics.get('ic_std') else "  IC Std: N/A")
        print(f"  IC IR: {metrics.get('ic_ir', 'N/A'):.4f}" if metrics.get('ic_ir') else "  IC IR: N/A")
        print(f"  P-value: {metrics.get('p_value', 'N/A'):.4f}" if metrics.get('p_value') else "  P-value: N/A")
        print(f"  Positive IC Ratio: {metrics.get('positive_ic_ratio', 'N/A'):.2%}" if metrics.get('positive_ic_ratio') else "  Positive IC Ratio: N/A")
    
    if result.get("saved_path"):
        print(f"\nResults saved to: {result['saved_path']}")
    
    print("\n" + result.get("message", ""))

# Example usage (uncomment after running the workflow)
# interpret_ic_results(result)

<a id="analyzing-factors"></a>
## 5.2) Analyzing Generated Factors

Let's look at examples of saved factor results:

In [ ]:
import json
from pathlib import Path
import os

# List saved factors (assumes we're in project root from cell 27's %cd ..)
output_dir = Path("src/factor_mining_workflow/output")
if output_dir.exists():
    factor_files = list(output_dir.glob("factor_*.json"))
    print(f"Found {len(factor_files)} saved factors:\n")
    
    for f in factor_files:  # Show first 3
        print(f"\n{'='*60}")
        print(f"File: {f.name}")
        print("="*60)
        
        with open(f, "r") as file:
            data = json.load(file)
            print(f"Timestamp: {data.get('timestamp', 'N/A')}")
            print(f"Iteration: {data.get('iteration', 'N/A')}")
            
            metrics = data.get('evaluation_metrics', {})
            if metrics:
                print(f"Mean IC: {metrics.get('mean_ic', 'N/A')}")
                print(f"P-value: {metrics.get('p_value', 'N/A')}")
else:
    print("No saved factors found. Run the workflow first!")

Found 9 saved factors:


File: factor_20260203_224353_iter3.json
Timestamp: 20260203_224353
Iteration: 3
Mean IC: 0.00514029262593512
P-value: 0.15077686173154903

File: factor_20260203_225357_iter1.json
Timestamp: 20260203_225357
Iteration: 1
Mean IC: -0.008101851407789767
P-value: 0.0008689790185862911

File: factor_20260203_235636_iter1.json
Timestamp: 20260203_235636
Iteration: 1
Mean IC: -0.003063733161265774
P-value: 0.07921501040867884

File: factor_20260204_004125_iter3.json
Timestamp: 20260204_004125
Iteration: 3
Mean IC: -0.009346960509742496
P-value: 0.00016183403826408593

File: factor_20260204_004407_iter3.json
Timestamp: 20260204_004407
Iteration: 3
Mean IC: -0.0040132838831864835
P-value: 0.07718153092827462

File: factor_20260204_004911_iter1.json
Timestamp: 20260204_004911
Iteration: 1
Mean IC: -0.013710019974056726
P-value: 2.418123799197147e-06

File: factor_20260204_005642_iter3.json
Timestamp: 20260204_005642
Iteration: 3
Mean IC: -0.005392609789735757
P-value: 0.0

<a id="next-steps"></a>
# 6) Next Steps

Now that you understand the factor mining workflow, here are some ideas for extending it:

1. **Add more operators**: Extend `calculator.json` with additional technical indicators
2. **Use different data**: Replace SP500 data with your own stock universe
3. **Customize evaluation**: Modify IC thresholds or add additional metrics like turnover
4. **Add more data fields**: Include fundamental data like earnings, book value, etc.
5. **Build factor portfolios**: Use accepted factors to construct trading strategies

## Additional Resources

- [NeMo Agent Toolkit Documentation](https://docs.nvidia.com/nemo-agent-toolkit/)